<a href="https://colab.research.google.com/github/fercbrt/03MIAR-Algoritmos-de-Optimizacion/blob/main/TRABAJO_PRACTICO/Trabajo_Pr%C3%A1ctico_Algoritmos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico

Nombre y Apellidos: Fernando Calvino Balonero

Url: https://github.com/fercbrt/03MIAR-Algoritmos-de-Optimizacion/blob/main/TRABAJO_PRACTICO/Trabajo_Pr%C3%A1ctico_Algoritmos.ipynb

Google Colab: https://colab.research.google.com/drive/1DmqeUubXU8ZTLvgr9LrIYopW8g_P0eKO?usp=sharing

## Organizar sesiones de doblaje

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en
las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el
estudio de grabación independientemente del número de tomas que se graben. No es
posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de
manera que el gasto por los servicios de los actores de doblaje sea el menor posible.

Para llevar a cabo la resolución del problema, se parte de un dataset en formato csv (`datos.csv`) que contiene la información de las actores que participan por cada toma. Los valores de la tabla tomarán un valor de 1 si el actor participa en la toma y un valor de 0 en caso contrario.

- Número de actores (columnas): 10
- Número de tomas (filas): 30

# Librerías necesarias
A continuación se procede a la imporatación de las librerías necesarias para la práctica.

In [3]:
import numpy as np
import pandas as pd
import random, copy, time

# Carga de datos y preprocesamiento

Antes de trabajar con los datos, es necesario cargarlos y preprocesarlos para que estén en un formato adecuado para el análisis. A continuación, se muestra cómo cargar el dataset `datos.csv` utilizando la biblioteca `pandas` en Python.

In [4]:
df = pd.read_csv("datos.csv", sep=";")
display(df.tail())

,Toma,1,2,3,4,5,6,7,8,9,10,Total
26,27,0,0,0,1,1,0,0,0,0,0,2.0
27,28,1,0,0,1,0,0,0,0,0,0,2.0
28,29,1,0,0,0,1,1,0,0,0,0,3.0
29,30,1,0,0,1,0,0,0,0,0,0,2.0
30,TOTAL,22,14,13,15,11,8,3,4,2,2,NaN


Tal y como se describe en el enunciado, cada fila del dataset representa una toma y cada columna representa un actor. El valor de cada celda indica si el actor participa (1) o no (0) en esa toma.

Además, se incluye una fila adicional al final del dataset que muestra el total de participaciones de cada actor en las tomas. Así como una columna adicional al final de cada fila que muestra el total de actores que participan en cada toma. Esta información puede ser útil para el análisis y la planificación de las sesiones de doblaje.

A continuación se procede a almacenar la información del dataset en tres variables:
- datos: una matriz que contiene la información de participación de los actores en las tomas.
- total_por_toma: un vector que contiene el total de actores que participan en cada toma.
- total_por_actor: un vector que contiene el total de participaciones de cada actor en las tomas.

In [5]:
datos = df.iloc[:-1, 1:-1].values
total_por_toma = df.iloc[:-1, -1].values
total_por_actor = df.iloc[-1, 1:-1].values

# Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

El espacio de soluciones se puede representar como un conjunto de combinaciones de tomas que cumplen con la restricción de no superar las 6 tomas por día. Cada solución representará una posible planificación de las sesiones de doblaje.

La función objetivo es minimizar el costo total de los actores de doblaje, que se calcula como el número de días necesarios para completar las sesiones de doblaje multiplicado por el costo diario de cada actor. Dado que el costo diario es constante para todos los actores, la función objetivo se puede simplificar a minimizar el número de días necesarios.

Las restricciones del problema incluyen:
1. No se pueden grabar más de 6 tomas por día.
2. Los actores deben coincidir en las tomas en las que sus personajes aparecen juntos. Esto significa que si dos actores participan en la misma toma, esa toma debe ser programada en el mismo día para ambos actores.

En primer lugar se definen las variables globales para establece las restricciones y tamaño de los datos

In [6]:
MAX_TOMAS_DIA = 6
N_TOMAS = datos.shape[0]
N_ACTORES = datos.shape[1]

A continuación se definen las funciones encargadas de representar el espacio de soluciones

In [7]:
def sesiones_a_vector(sesiones):
    v = np.empty(N_TOMAS, dtype=int)
    for d, dia in enumerate(sesiones):
        for t in dia:
            v[t] = d
    return v

def vector_a_sesiones(x):
    n_dias = x.max() + 1
    return [[t for t in range(N_TOMAS) if x[t] == d] for d in range(n_dias)]

La función objetivo es definida como el coste de sumar el número de actores distintos que trabajan un mismo día.

In [8]:
def funcion_objetivo(sesiones):
    coste = 0
    for dia in sesiones:
        if dia:
            actores_dia = np.any(datos[dia, :], axis=0)
            coste += int(actores_dia.sum())
    return coste

La siguiente función implementa la restricción que impide grabar más de 6 tomas por día.

In [9]:
def es_factible(sesiones):
    return all(len(dia) <= MAX_TOMAS_DIA for dia in sesiones)

La restricción que obliga a todos los actores que participan en una toma estén presentes el mismo día se cumple de forma implícita a la hora de construir la solución.

# Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones


El problema de planificar las sesiones de doblaje es equivalente al Bin Packing con función de coste, que pertenece a la clase NP-Hard. Esto implica que no existe ningún algoritmo de tiempo polinomial conocido para resolverlo de forma exacta.

**Espacio de soluciones:**

Una solución se representa como un vector de asignación $x = (x_1, \ldots, x_n)$ donde $x_i \in \{1, \ldots, D\}$ indica el día asignado a la toma $i$, sujeto a que cada día acumula a lo sumo $c = 6$ tomas.

- Número mínimo de días necesarios: $D_{\min} = \lceil n/c \rceil = \lceil 30/6 \rceil = 5$
- Número máximo de días: $D_{\max} = n = 30$ (una toma por día)

**Cota superior del espacio de soluciones**

$$|\text{Espacio}| \leq D_{\max}^n = 30^{30} \approx 2{,}06 \times 10^{44}$$

**Contabilización exacta con $D_{\min}$ días y grupos iguales**

$$\frac{n!}{(c!)^{D_{\min}}} = \frac{30!}{(6!)^5} \approx 1{,}37 \times 10^{18}$$


# Diseño
- ¿Que técnica utilizo? ¿Por qué?

La técnica seleccionada para resolver este problema será el Algoritmo de Optimización por Colonia de Hormigas (ACO, por sus siglas en inglés), visto anteriormente en esta misma asignatura.

## Por qué utilizar ACO

El uso de ACO para la resolución del problema de doblaje se debe a varios factores.

- **Problema de asignación combinatoria**. El problema de asignar tomas a días es un problema NP-Duro similar al Bin Packing. ACO es especialmente efectivo para problemas de optimización combinatoria donde se busca encontrar buenas soluciones en un espacio de búsqueda muy grande.

- **Balance entre exploración y explotación**. ACO explora múltiples soluciones mediante hormigas que construyen rutas probabilísticamente, guiadas por feromonas (memoria de buenas soluciones) y heurística local, permitiendo escapar de óptimos locales.

- **Adaptabilidad**. Las feromonas se actualizan dinámicamente, reforzando las buenas asignaciones de tomas a días, lo que permite al algoritmo converger hacia soluciones de calidad.

## Algoritmo Voraz como punto de partida

El algoritmo voraz construye una solución inicial siguiendo esta estrategia:

1. Ordenar las tomas por número de actores que participan (de mayor a menor)
2. Para cada toma, asignarla al primer día disponible que:
   - Tenga capacidad (menos de 6 tomas)
   - Minimice el número de actores nuevos que se incorporan ese día
3. Si no hay día disponible, crear uno nuevo

Esta heurística tiende a agrupar tomas con actores comunes, reduciendo el coste total.

In [10]:
D_MAX_DIAS = int(np.ceil(N_TOMAS / MAX_TOMAS_DIA)) + 1

# Función para calcular el coste del problema
def _coste_incremental(actores_t, actores_dia):
    return int((actores_t & ~actores_dia).sum())


def _greedy_por_orden(orden_tomas):
    sesiones = []
    act_x_dia = []

    for t in orden_tomas:
        actores_t = datos[t, :].astype(bool)

        mejor_d = None
        mejor_key = None

        for d, dia in enumerate(sesiones):
            if len(dia) >= MAX_TOMAS_DIA:
                continue

            inc = _coste_incremental(actores_t, act_x_dia[d])
            overlap = int((actores_t & act_x_dia[d]).sum())

            key = (inc, -overlap, -len(dia), d)
            if (mejor_key is None) or (key < mejor_key):
                mejor_key = key
                mejor_d = d

        if mejor_d is None:
            sesiones.append([t])
            act_x_dia.append(actores_t.copy())
        else:
            sesiones[mejor_d].append(t)
            act_x_dia[mejor_d] |= actores_t

    coste = funcion_objetivo(sesiones)
    return sesiones, act_x_dia, coste


def greedy_solucion():
    base = np.arange(N_TOMAS)

    ordenes = [
        list(np.argsort(-total_por_toma)),
        list(np.argsort(-total_por_toma + 0.01 * base)),
        list(np.argsort(-total_por_toma - 0.01 * base)),
        list(np.argsort([-int(total_por_toma[t]) for t in range(N_TOMAS)])),
    ]

    mejor = None
    for orden_local in ordenes:
        ses, act, c = _greedy_por_orden(orden_local)
        if (mejor is None) or (c < mejor[2]):
            mejor = (ses, act, c)

    return mejor


sol_g, act_g, coste_g = greedy_solucion()
orden = list(np.argsort(-total_por_toma))

print("Solución Voraz")
for i, dia in enumerate(sol_g):
    actores_dia = [int(a) + 1 for a in np.where(np.any(datos[dia, :], axis=0))[0]]
    tomas_dia = sorted([int(t) + 1 for t in dia])
    print(f"  Día {i+1:2d}: tomas {tomas_dia} | actores {actores_dia} | coste: {len(actores_dia)}")
print(f"  Coste greedy: {coste_g}")

Solución Voraz
  Día  1: tomas [1, 4, 7, 10, 11, 12] | actores [1, 2, 3, 4, 5, 6, 7, 8, 9] | coste: 9
  Día  2: tomas [6, 9, 20, 22, 25, 26] | actores [1, 2, 3, 4, 5, 9, 10] | coste: 7
  Día  3: tomas [2, 3, 5, 8, 15, 29] | actores [1, 2, 3, 4, 5, 6, 7, 8] | coste: 8
  Día  4: tomas [13, 14, 16, 17, 18, 19] | actores [1, 3, 4, 5, 6, 10] | coste: 6
  Día  5: tomas [21, 23, 24, 27, 28, 30] | actores [1, 3, 4, 5, 6, 8] | coste: 6
  Coste greedy: 36


## Busqueda local

La búsqueda local sirve para refinar y mejorar cada solución que construye una hormiga antes de evaluar su coste.

Tomando una solución candidata, intenta mover tomas de un día para otro, calcula el coste para cada movimiento posible y si encuentra una mejora, aplica dicho movimiento, repitiendo el mismo procedimiento hasta que no pueda mejorar más su coste.

In [11]:
def busqueda_local(sesiones):
    ses = [list(d) for d in sesiones]

    cnt_dia = []
    coste_por_dia = []
    for dia in ses:
        cnt = datos[dia, :].sum(axis=0).astype(int) if dia else np.zeros(N_ACTORES, dtype=int)
        cnt_dia.append(cnt)
        coste_por_dia.append(int(np.count_nonzero(cnt)))

    coste_total = int(sum(coste_por_dia))

    mejoro = True
    while mejoro:
        mejoro = False
        mejor_delta = 0
        mejor_mov = None

        for d_orig, dia in enumerate(ses):
            if not dia:
                continue

            for t in dia:
                actores_t = datos[t, :].astype(bool)

                delta_out = -int(np.sum((cnt_dia[d_orig] == 1) & actores_t))

                for d_dest, dia_dest in enumerate(ses):
                    if d_dest == d_orig or len(dia_dest) >= MAX_TOMAS_DIA:
                        continue

                    delta_in = int(np.sum((cnt_dia[d_dest] == 0) & actores_t))
                    delta = delta_out + delta_in

                    if delta < mejor_delta:
                        mejor_delta = delta
                        mejor_mov = (t, d_orig, d_dest, actores_t)

        if mejor_mov is not None:
            t, d_orig, d_dest, actores_t = mejor_mov

            ses[d_orig].remove(t)
            ses[d_dest].append(t)

            cnt_dia[d_orig][actores_t] -= 1
            cnt_dia[d_dest][actores_t] += 1

            coste_por_dia[d_orig] = int(np.count_nonzero(cnt_dia[d_orig]))
            coste_por_dia[d_dest] = int(np.count_nonzero(cnt_dia[d_dest]))
            coste_total += mejor_delta

            nuevos_ses = []
            nuevos_cnt = []
            nuevos_costes = []
            for i in range(len(ses)):
                if ses[i]:
                    nuevos_ses.append(ses[i])
                    nuevos_cnt.append(cnt_dia[i])
                    nuevos_costes.append(coste_por_dia[i])
            ses, cnt_dia, coste_por_dia = nuevos_ses, nuevos_cnt, nuevos_costes

            mejoro = True

    return ses, coste_total

## Implementación de ACO

Cada hormiga construye una solución completa asignando iterativamente cada toma a un día:

$$P(t \to d) = \frac{\tau_{t,d}^\alpha \cdot \eta_{t,d}^\beta}{\sum_{d'} \tau_{t,d'}^\alpha \cdot \eta_{t,d'}^\beta}$$

donde:
- $\tau_{t,d}$ = feromona en el arco (toma $t$, día $d$) — aprendizaje acumulado
- $\eta_{t,d} = \frac{1}{1 + \Delta\text{coste}(t \to d)}$ — información heurística (inversa del coste incremental)
- $\alpha$ = importancia de la feromona
- $\beta$ = importancia de la heurística

**Actualización de feromonas** tras cada iteración:
1. **Evaporación:** $\tau_{t,d} \leftarrow (1 - \rho)\,\tau_{t,d}$
2. **Depósito:** la mejor hormiga de la iteración deposita $\Delta\tau = Q / \text{coste}$ en cada arco $(t,d)$ de su solución

Adicionalmente se aplica la misma **búsqueda local** del VNS (*relocate best-improvement*) sobre la solución de cada hormiga antes de evaluar, convirtiendo el esquema en **ACO + LS**.


In [12]:
# Fijamos una semilla
random.seed(0)
np.random.seed(0)

# Parámetros ACO
N_HORMIGAS  = 20      # hormigas por iteración
ALPHA       = 1.0     # peso feromona
BETA        = 3.0     # peso heurística
RHO         = 0.15    # tasa de evaporación
Q           = 1.0     # constante de depósito
TAU_INIT    = 1.0     # feromona inicial
TAU_MIN     = 1e-4    # feromona mínima (evita extinción)
LIMIT_ACO   = 5      # segundos

### Configuración de parámetros ACO

Los parámetros mostrados arriba controlan el comportamiento del algoritmo:

- **N_HORMIGAS**: número de soluciones candidatas generadas por iteración
- **ALPHA** y **BETA**: controlan la importancia relativa de feromonas vs. heurística
- **RHO**: tasa de evaporación de feromonas (olvido del pasado)
- **Q**: factor de depósito de feromonas
- **TAU_INIT** y **TAU_MIN**: valores iniciales y mínimos de feromona
- **LIMIT_ACO**: tiempo máximo de ejecución en segundos

In [13]:
# Inicializar feromonas τ[toma][día_relativo 0..D_MAX_DIAS-1]
D_SLOTS = D_MAX_DIAS
tau     = np.full((N_TOMAS, D_SLOTS), TAU_INIT)

# Heurística η: inversa del coste incremental
def eta(t_idx, d_actores, dia_toma):
    actores_t = datos[t_idx, :].astype(bool)
    if dia_toma is None:
        inc = int(actores_t.sum())
    else:
        inc = int((actores_t & ~d_actores).sum())
    return 1.0 / (1.0 + inc)

### Construcción de soluciones por las hormigas

Cada hormiga construye una solución asignando iterativamente cada toma a un día. La probabilidad de asignar la toma $t$ al día $d$ se calcula como:

$$P(t \to d) = \frac{\tau_{t,d}^\alpha \cdot \eta_{t,d}^\beta}{\sum_{d'} \tau_{t,d'}^\alpha \cdot \eta_{t,d'}^\beta}$$

donde la **heurística** $\eta$ representa el inverso del coste incremental de agregar la toma $t$ al día $d$.

In [14]:
# Construcción de solución por una hormiga
def construir_solucion():
    sesiones = []
    act_dia  = []

    for t in orden:
        actores_t  = datos[t, :].astype(bool)
        candidatos = []

        for d, dia in enumerate(sesiones):
            if len(dia) < MAX_TOMAS_DIA:
                candidatos.append((d, act_dia[d], False))

        if len(sesiones) < D_SLOTS:
            candidatos.append((len(sesiones), None, True))

        if not candidatos:
            candidatos.append((len(sesiones), None, True))

        pesos = []
        for (d_idx, d_act, nuevo) in candidatos:
            d_col = min(d_idx, D_SLOTS - 1)
            fen   = tau[t, d_col] ** ALPHA
            heu   = eta(t, d_act, d_act) ** BETA
            pesos.append(fen * heu)

        total = sum(pesos)
        if total == 0:
            total = 1.0
        probs = [p / total for p in pesos]

        r, acc, elegido = random.random(), 0.0, candidatos[-1]
        for cand, p in zip(candidatos, probs):
            acc += p
            if r <= acc:
                elegido = cand
                break

        d_idx, d_act, nuevo = elegido
        if nuevo:
            sesiones.append([t])
            act_dia.append(actores_t.copy())
        else:
            act_dia[d_idx] |= actores_t
            sesiones[d_idx].append(t)

    return sesiones

### Actualización de feromonas

Tras cada iteración, las feromonas se actualizan en dos pasos:

1. **Evaporación**: $\tau_{t,d} \leftarrow (1 - \rho)\,\tau_{t,d}$ — reduce todas las feromonas, permitiendo que el algoritmo olvide soluciones pasadas
2. **Depósito**: la mejor solución de la iteración deposita $\Delta\tau = Q / \text{coste}$ en cada arco $(t,d)$ de su ruta, reforzando las buenas asignaciones

Los valores se mantienen dentro de un rango $[\tau_{\min}, \infty)$ para evitar que las feromonas desaparezcan completamente.

In [15]:
# Actualización de feromonas
def actualizar_tau(mejor_iter_ses, mejor_iter_coste):
    global tau
    tau *= (1.0 - RHO)
    deposito = Q / mejor_iter_coste
    for d_idx, dia in enumerate(mejor_iter_ses):
        d_col = min(d_idx, D_SLOTS - 1)
        for t in dia:
            tau[t, d_col] += deposito
    np.clip(tau, TAU_MIN, None, out=tau)

### Ejecución del algoritmo ACO

El bucle principal ejecuta iteraciones durante un tiempo máximo (`LIMIT_ACO` segundos). En cada iteración:

1. Se generan `N_HORMIGAS` soluciones candidatas mediante la función `construir_solucion()`
2. Cada solución se mejora con **búsqueda local** mediante relocate best-improvement
3. Se registra la mejor solución de la iteración y se actualiza el incumbente global si es necesario
4. Se actualizan las feromonas para reforzar la mejor solución encontrada

In [16]:
# Bucle principal ACO
mejor_aco = {'coste': coste_g, 'sesiones': [list(d) for d in sol_g]}
iters_aco = 0
t_aco     = time.time()

print("ACO – ejecutando...\n")

while time.time() - t_aco < LIMIT_ACO:
    iters_aco += 1
    mejor_iter_coste = float('inf')
    mejor_iter_ses   = None

    for _ in range(N_HORMIGAS):
        ses      = construir_solucion()
        ses, c   = busqueda_local(ses)

        if c < mejor_iter_coste:
            mejor_iter_coste = c
            mejor_iter_ses   = copy.deepcopy(ses)

        if c < mejor_aco['coste']:
            mejor_aco['coste']    = c
            mejor_aco['sesiones'] = copy.deepcopy(ses)
            print(f"  iter {iters_aco:4d}  ► nuevo mejor: {mejor_aco['coste']}")

    actualizar_tau(mejor_iter_ses, mejor_iter_coste)

elapsed_aco = time.time() - t_aco

ACO – ejecutando...

  iter    1  ► nuevo mejor: 35
  iter    1  ► nuevo mejor: 33
  iter    1  ► nuevo mejor: 28


### Visualización de resultados

A continuación se muestran los detalles de la solución encontrada por ACO:
- Para cada día se indica el número de día, las tomas asignadas, los actores necesarios y su costo
- Se compara el coste del algoritmo voraz inicial con el resultado final de ACO

In [17]:
# Resultados
print(f"\nSolución ACO  [tiempo: {elapsed_aco:.2f}s | iteraciones: {iters_aco:,}]")
for i, dia in enumerate(mejor_aco['sesiones']):
    act   = [int(a)+1 for a in np.where(np.any(datos[dia, :], axis=0))[0]]
    tomas = sorted([int(t)+1 for t in dia])
    print(f"  Día {i+1:2d}: tomas {tomas} | actores {act} | coste: {len(act)}")

print(f"  Coste ACO: {mejor_aco['coste']}")

print("\n╔══════════════════════╦════════╦══════════╗")
print("║ Algoritmo            ║ Coste  ║ Tiempo   ║")
print("╠══════════════════════╬════════╬══════════╣")
print(f"║ Voraz                ║  {coste_g:>4d}  ║    —     ║")
print(f"║ ACO  ({LIMIT_ACO:>3d}s)          ║  {mejor_aco['coste']:>4d}  ║ {elapsed_aco:>6.2f}s  ║")
print("╚══════════════════════╩════════╩══════════╝")


Solución ACO  [tiempo: 5.01s | iteraciones: 118]
  Día  1: tomas [1, 5, 6, 7, 11, 20] | actores [1, 2, 3, 4, 5, 8] | coste: 6
  Día  2: tomas [9, 10, 12, 22, 25, 26] | actores [1, 2, 3, 4, 5, 6, 9, 10] | coste: 8
  Día  3: tomas [3, 4, 8, 15, 21, 29] | actores [1, 2, 5, 6, 7, 8] | coste: 6
  Día  4: tomas [14, 17, 18, 19, 23, 24] | actores [1, 3, 6] | coste: 3
  Día  5: tomas [2, 13, 16, 27, 28, 30] | actores [1, 3, 4, 5, 10] | coste: 5
  Coste ACO: 28

╔══════════════════════╦════════╦══════════╗
║ Algoritmo            ║ Coste  ║ Tiempo   ║
╠══════════════════════╬════════╬══════════╣
║ Voraz                ║    36  ║    —     ║
║ ACO  (  5s)          ║    28  ║   5.01s  ║
╚══════════════════════╩════════╩══════════╝


# Análisis de Complejidad del Algoritmo ACO

## Notación y variables

Sea:
- $n = $ número de tomas
- $a = $ número de actores
- $h = $ número de hormigas por iteración
- $d = $ número máximo de días (función de $n$ y capacidad por día)
- $I = $ número total de iteraciones ejecutadas
- $t = $ tiempo límite

## Complejidad de operaciones elementales

### 1. Función `construir_solucion()`

Para cada toma $t$ (hay $n$ tomas):
- Se itera sobre los $d$ días existentes: $O(d)$
- Para cada candidato se calcula feromona y heurística: $O(a)$ operaciones bit-a-bit
- Se elige probabilísticamente: $O(d)$

**Complejidad de `construir_solucion()`: $O(n \cdot d \cdot a)$**

Dado que $d \ll n$ en la mayoría de casos, se simplifica a $O(n \cdot a)$ en la práctica.

### 2. Función `busqueda_local()`

**Inicialización**: $O(d \cdot a)$ para construir contadores de actores por día

**Bucle de mejora** (puede repetirse múltiples veces):
- Para cada día: $O(d)$
- Para cada toma en el día: $O(n)$ en el peor caso
- Para cada día destino: $O(d)$
- Operación lógica con actores: $O(a)$

En cada iteración del bucle de mejora se intenta evaluar $O(n \cdot d)$ movimientos, cada uno con costo $O(a)$.

**Complejidad en el peor caso: $O(n \cdot d \cdot a \cdot M)$**

donde $M$ es el número de iteraciones del bucle de mejora. En la práctica, $M$ es pequeño porque el número de mejoras posibles es limitado.

**Complejidad promedio: $O(n \cdot d \cdot a)$**

### 3. Función `actualizar_tau()`

- Multiplicación escalar de matriz: $O(n \cdot d)$ [evaporación]
- Iteración sobre días y tomas: $O(d \cdot \lceil n/d \rceil) = O(n)$ [depósito]
- Limitación de valores: $O(n \cdot d)$

**Complejidad de `actualizar_tau()`: $O(n \cdot d)$**

## Complejidad por iteración

Una iteración ejecuta:
- $h$ construcciones de solución: $h \cdot O(n \cdot a)$
- $h$ búsquedas locales: $h \cdot O(n \cdot d \cdot a)$
- 1 actualización de feromonas: $O(n \cdot d)$

**Complejidad por iteración: $O(h \cdot n \cdot d \cdot a + n \cdot d) = O(h \cdot n \cdot d \cdot a)$**

(ya que $h \gg 1$)

## Complejidad total del algoritmo ACO

El número de iteraciones $I$ depende del tiempo límite y del tiempo requerido por cada iteración:

$$I = \left\lfloor \frac{t}{\text{tiempo\_por\_iteración}} \right\rfloor$$

**Complejidad total: $O(I \cdot h \cdot n \cdot d \cdot a)$**

## Resumen comparativo

| Operación | Complejidad |
|-----------|------------|
| `construir_solucion()` | $O(n \cdot a)$ |
| `busqueda_local()` | $O(n \cdot d \cdot a)$ * |
| `actualizar_tau()` | $O(n \cdot d)$ |
| Una iteración ACO | $O(h \cdot n \cdot d \cdot a)$ |
| **ACO completo** | $O(I \cdot h \cdot n \cdot d \cdot a)$ |

*Con $M$ iteraciones de mejora internas limitadas por la estructura del problema.